# Visualização de features da VGG16

Carrega um modelo treinado pelo `base.ipynb` e gera:

1. Filtros da primeira camada conv.
2. Mapas de ativação de uma imagem em cada bloco.
3. Imagens do teste que mais ativam cada canal (*dataset examples*).
4. Feature visualization por otimização (canais e classes).
5. Grad-CAM.
6. Embeddings da penúltima camada (PCA e t-SNE).

**Como usar:** edite apenas a célula abaixo e rode tudo (*Run All*).

In [ ]:
# ======================= CONFIGURAÇÃO =======================
# Qual modelo carregar: ./USUARIO/TESTE/<classes>/model.pt
USUARIO = "alvaro"
TESTE = "teste01"
CLASSES = ["apple", "orange", "pear"]
#   mesma especificação usada no base.ipynb (nomes, índices, int N ou None).
#   None -> usa a única run existente em ./USUARIO/TESTE/ (ou lista as disponíveis).
SEED = 42                  # só importa se CLASSES for um inteiro N (sorteio)

# Camada analisada nas seções 2-4: conv1_1 ... conv5_3
CAMADA = "conv4_3"
CANAIS = None              # lista de canais (ex.: [3, 17, 42]); None -> os N_CANAIS mais ativos
N_CANAIS = 12

IMAGEM = 0                 # índice da imagem (no teste do subconjunto) para os mapas de ativação
N_IMAGENS_GRADCAM = 8
CAMADA_GRADCAM = "conv4_3" # no modelo "cifar" (32x32): conv4_x = 4x4, conv5_x = 2x2

PASSOS_OTIMIZACAO = 256    # iterações da feature visualization
DISPOSITIVO = "auto"       # "auto" | "cuda" | "cuda:1" | "mps" | "cpu"
SALVAR_FIGURAS = True      # salva PNGs em <pasta da run>/figuras/
# ============================================================

---

*Daqui para baixo não é preciso editar nada.*

## 1. Carregar modelo e dados

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from tqdm.auto import tqdm

from common import conv_layers, eval_transform, find_run_dir, get_device, load_run, norm_stats, set_seed

set_seed(SEED)
device = get_device(DISPOSITIVO)
run_dir = find_run_dir(USUARIO, TESTE, CLASSES, seed=SEED)
model, ckpt = load_run(run_dir, device)
cfg, selected, class_names = ckpt["cfg"], ckpt["selected"], ckpt["class_names"]
NUM_CLASSES = len(selected)

# ReLU in-place sobrescreveria a saída do BN capturada pelos hooks.
for m in model.modules():
    if isinstance(m, nn.ReLU):
        m.inplace = False
for p in model.parameters():
    p.requires_grad_(False)

fig_dir = run_dir / "figuras"
if SALVAR_FIGURAS:
    fig_dir.mkdir(exist_ok=True)

def savefig(fig, name):
    if SALVAR_FIGURAS:
        fig.savefig(fig_dir / f"{name}.png", dpi=150, bbox_inches="tight")

print("run:", run_dir)
print("device:", device, "| variante:", cfg["model_variant"], f"| época {ckpt['epoch']}, val acc {ckpt['val_acc']:.3f}")
print("classes:", class_names)

In [ ]:
orig_to_new = {o: n for n, o in enumerate(selected)}
test_full = datasets.CIFAR100(cfg["data_dir"], train=False, transform=eval_transform(cfg),
                              target_transform=lambda y: orig_to_new.get(y, -1))
test_idx = np.flatnonzero(np.isin(test_full.targets, selected)).tolist()
test_ds = Subset(test_full, test_idx)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

MEAN, STD = (torch.tensor(s, device=device).view(1, 3, 1, 1) for s in norm_stats(cfg["model_variant"]))
IMG_SIZE = cfg["img_size"]

def denorm(x):
    """Tensor normalizado (C,H,W) ou (N,C,H,W) -> imagem em [0,1] (H,W,C) numpy."""
    x = x.detach().to(device)
    single = x.dim() == 3
    x = (x[None] if single else x) * STD + MEAN
    x = x.clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()
    return x[0] if single else x

def spotlight(img, heat, floor=0.2):
    """Escurece a imagem onde o mapa `heat` (em [0,1]) é baixo — legível sobre qualquer cor."""
    return img * (floor + (1 - floor) * heat[..., None])

LAYERS = conv_layers(model)
if CAMADA not in LAYERS or CAMADA_GRADCAM not in LAYERS:
    raise ValueError(f"Camadas válidas: {list(LAYERS)}")

print(f"teste: {len(test_ds)} imagens")

### Hooks

`Capture` guarda as saídas de camadas durante o forward. Para cada camada `convX_Y` há duas versões:
`"relu"` (ativação, ≥ 0) e `"bn"` (pré-ativação, usada na otimização para não zerar gradientes).

In [ ]:
class Capture:
    """Context manager: captura saídas de camadas nomeadas. Uso:
        with Capture(model, ["conv3_3"]) as cap:
            model(x)
        cap["conv3_3"]  # (N, C, H, W)
    """

    def __init__(self, model, names, kind="relu", detach=True):
        self.names, self.kind, self.detach, self.out, self.handles = names, kind, detach, {}, []

    def __enter__(self):
        idx = {"conv": 0, "bn": 1, "relu": 2}[self.kind]
        for name in self.names:
            mod = LAYERS[name][idx]
            self.handles.append(mod.register_forward_hook(self._hook(name)))
        return self

    def _hook(self, name):
        def fn(_m, _i, o):
            self.out[name] = o.detach() if self.detach else o
        return fn

    def __exit__(self, *exc):
        for h in self.handles:
            h.remove()

    def __getitem__(self, name):
        return self.out[name]


with torch.no_grad(), Capture(model, list(LAYERS)) as cap:
    model(test_ds[0][0][None].to(device))
print(f"{'camada':<8} {'canais':>6}  mapa")
for name in LAYERS:
    _, c, h, w = cap[name].shape
    print(f"{name:<8} {c:>6}  {h}x{w}")

## 2. Filtros da primeira camada (`conv1_1`)

Cada filtro 3×3×3 normalizado individualmente para [0, 1] e exibido como RGB.

In [ ]:
w = LAYERS["conv1_1"][0].weight.detach().cpu()           # (64, 3, 3, 3)
w = (w - w.amin((1, 2, 3), keepdim=True)) / (w.amax((1, 2, 3), keepdim=True) - w.amin((1, 2, 3), keepdim=True) + 1e-8)
n = w.shape[0]
cols = 16
fig, axes = plt.subplots(int(np.ceil(n / cols)), cols, figsize=(cols * 0.6, np.ceil(n / cols) * 0.6))
for i, ax in enumerate(axes.flat):
    ax.axis("off")
    if i < n:
        ax.imshow(w[i].permute(1, 2, 0), interpolation="nearest")
fig.suptitle("conv1_1: filtros", y=1.02)
savefig(fig, "filtros_conv1_1")
plt.show()

## 3. Mapas de ativação de uma imagem

Para a imagem `IMAGEM`, mostra os 8 canais mais ativos (maior média) da última conv de cada bloco,
e depois todos os canais de `CAMADA` ordenados por ativação.

In [ ]:
x, y = test_ds[IMAGEM]
with torch.no_grad(), Capture(model, list(LAYERS)) as cap:
    logits = model(x[None].to(device))
pred = logits.argmax(1).item()
prob = logits.softmax(1)[0, pred].item()

block_last = [n for n in LAYERS if not any(o.startswith(n[:6]) and o > n for o in LAYERS)]  # conv1_2, ..., conv5_3
top = 8
fig, axes = plt.subplots(len(block_last), top + 1, figsize=((top + 1) * 1.2, len(block_last) * 1.3))
for r, name in enumerate(block_last):
    a = cap[name][0].cpu()                              # (C, H, W)
    order = a.mean((1, 2)).argsort(descending=True)[:top]
    axes[r, 0].imshow(denorm(x))
    axes[r, 0].set_ylabel(f"{name}\n{a.shape[1]}x{a.shape[2]}", fontsize=8)
    for c, ch in enumerate(order):
        ax = axes[r, c + 1]
        ax.imshow(a[ch], cmap="gray", interpolation="nearest")
        ax.set_title(f"#{ch.item()}", fontsize=7)
    for ax in axes[r]:
        ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"real: {class_names[y]}  |  predito: {class_names[pred]} ({prob:.2f})", y=1.01)
plt.tight_layout()
savefig(fig, f"ativacoes_img{IMAGEM}")
plt.show()

In [ ]:
a = cap[CAMADA][0].cpu()
order = a.mean((1, 2)).argsort(descending=True)
n_show = min(64, a.shape[0])
cols = 16
rows = int(np.ceil(n_show / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 0.8, rows * 0.9))
vmax = a.max().item()
for i, ax in enumerate(axes.flat):
    ax.axis("off")
    if i < n_show:
        ch = order[i].item()
        ax.imshow(a[ch], cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
        ax.set_title(f"#{ch}", fontsize=6)
active = (a.amax((1, 2)) > 0).float().mean().item()
fig.suptitle(f"{CAMADA}: {n_show} canais mais ativos (escala comum) | {active:.0%} dos canais com alguma ativação", y=1.02)
savefig(fig, f"ativacoes_{CAMADA}_img{IMAGEM}")
plt.show()

## 4. Imagens que mais ativam cada canal

Passa todo o teste do subconjunto pela rede e, para cada canal de `CAMADA`, guarda a ativação
máxima espacial. Mostra as 8 imagens com maior ativação, com o mapa do canal sobreposto.
As regiões escurecidas são onde o canal pouco ativa. Também mostra a distribuição de classes dessas imagens — canais "seletivos" concentram-se em uma classe.

In [ ]:
acts_max = []   # (N, C): máximo espacial por canal
with torch.no_grad(), Capture(model, [CAMADA]) as cap:
    for xb, _ in tqdm(test_loader, desc="ativações"):
        model(xb.to(device))
        acts_max.append(cap[CAMADA].amax((2, 3)).cpu())
acts_max = torch.cat(acts_max)
labels = torch.tensor([orig_to_new[test_full.targets[i]] for i in test_idx])

if CANAIS is None:
    canais = acts_max.mean(0).argsort(descending=True)[:N_CANAIS].tolist()
else:
    canais = list(CANAIS)
print("canais:", canais)

# seletividade: fração das top-50 imagens que pertencem à classe mais frequente
k_sel = min(50, len(test_ds))
selectivity = {}
for ch in range(acts_max.shape[1]):
    top_lbl = labels[acts_max[:, ch].topk(k_sel).indices]
    counts = torch.bincount(top_lbl, minlength=NUM_CLASSES)
    selectivity[ch] = (counts.max().item() / k_sel, counts.argmax().item())

In [ ]:
k = 8
fig, axes = plt.subplots(len(canais), k + 1, figsize=((k + 1) * 1.1, len(canais) * 1.15),
                         gridspec_kw={"width_ratios": [1] * k + [1.6]})
for r, ch in enumerate(canais):
    top_idx = acts_max[:, ch].topk(k).indices
    xb = torch.stack([test_ds[i][0] for i in top_idx]).to(device)
    with torch.no_grad(), Capture(model, [CAMADA]) as cap:
        model(xb)
    maps = F.interpolate(cap[CAMADA][:, ch:ch + 1], size=IMG_SIZE, mode="bilinear", align_corners=False)
    maps = (maps / (maps.amax((2, 3), keepdim=True) + 1e-8)).squeeze(1).cpu().numpy()
    imgs = denorm(xb)
    for c in range(k):
        ax = axes[r, c]
        ax.imshow(spotlight(imgs[c], maps[c]))
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel(f"#{ch}", fontsize=8)
        ax.set_title(class_names[labels[top_idx[c]]][:10], fontsize=6)
    # distribuição de classes no top-50
    ax = axes[r, k]
    counts = torch.bincount(labels[acts_max[:, ch].topk(k_sel).indices], minlength=NUM_CLASSES).numpy()
    ax.barh(range(NUM_CLASSES), counts / k_sel, color="#2a78d6", height=0.7)
    ax.set_xlim(0, 1); ax.invert_yaxis()
    ax.set_yticks(range(NUM_CLASSES) if NUM_CLASSES <= 10 else [])
    ax.set_yticklabels(class_names if NUM_CLASSES <= 10 else [], fontsize=6)
    ax.tick_params(axis="x", labelsize=6)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[0, k].set_title(f"classes no top-{k_sel}", fontsize=7)
fig.suptitle(f"{CAMADA}: imagens do teste que mais ativam cada canal", y=1.005)
plt.tight_layout()
savefig(fig, f"top_imagens_{CAMADA}")
plt.show()

In [ ]:
sel = np.array([selectivity[c][0] for c in range(acts_max.shape[1])])
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.hist(sel, bins=np.linspace(0, 1, 21), color="#2a78d6", edgecolor="white", linewidth=2)
ax.axvline(1 / NUM_CLASSES, color="#52514e", linestyle="--", linewidth=1)
ax.text(1 / NUM_CLASSES, ax.get_ylim()[1] * 0.95, " acaso", fontsize=8, color="#52514e", va="top")
ax.set_xlabel(f"fração do top-{k_sel} na classe dominante")
ax.set_ylabel("canais")
ax.set_title(f"{CAMADA}: seletividade de classe dos {len(sel)} canais", fontsize=10)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
savefig(fig, f"seletividade_{CAMADA}")
plt.show()

print("canais mais seletivos:")
for ch in np.argsort(-sel)[:10]:
    print(f"  #{ch:<4} {sel[ch]:.2f}  -> {class_names[selectivity[ch][1]]}")

## 5. Feature visualization por otimização

Otimiza uma imagem (a partir de ruído) para maximizar um objetivo:
- **canal**: média da pré-ativação (saída do BN) do canal em `CAMADA`;
- **classe**: logit da classe.

Regularização (no estilo do Lucid / Olah et al. 2017):
- imagem parametrizada no **domínio de Fourier** com espectro escalado por 1/f^1.5 (favorece baixas frequências)
  e cores **decorrelacionadas**;
- renderizada em 2× a resolução e reduzida com antialias (atenua o padrão xadrez dos poolings);
- a cada passo, **transformações aleatórias** (padding, escala, deslocamento) — só sobrevivem padrões robustos.

Em 32×32 os resultados são pequenos e saturados; isso é esperado. Eles ficam mais claros em modelos bem treinados.

In [ ]:
# matriz de decorrelação de cor (Lucid, estimada no ImageNet)
_COLOR = torch.tensor([[0.26, 0.09, 0.02], [0.27, 0.00, -0.05], [0.27, -0.09, 0.03]])
COLOR_CORR = (_COLOR / _COLOR.norm(dim=0).max()).to(device)


def fourier_image(n, size, decay=1.5, seed=0):
    """Imagem parametrizada no espectro. Retorna (parâmetro, render() -> (n,3,size,size) em [0,1])."""
    g = torch.Generator().manual_seed(seed)
    fy = torch.fft.fftfreq(size)[:, None]
    fx = torch.fft.rfftfreq(size)[None, :]
    scale = (1.0 / torch.clamp(torch.sqrt(fx ** 2 + fy ** 2), min=1.0 / size) ** decay).to(device)
    spec = (torch.randn(n, 3, size, size // 2 + 1, 2, generator=g) * 0.01).to(device).requires_grad_(True)

    def render():
        x = torch.fft.irfftn(torch.view_as_complex(spec) * scale, s=(size, size), norm="ortho")
        return torch.sigmoid(torch.einsum("nchw,dc->ndhw", x, COLOR_CORR))

    return spec, render


def optimize_images(objective, n, steps=PASSOS_OTIMIZACAO, lr=0.02, decay=1.5, seed=0):
    """objective(logits, cap) -> tensor (n,) a maximizar. Retorna imagens (n, H, W, 3) em [0,1]."""
    rng = np.random.default_rng(seed)
    up = 2 if IMG_SIZE <= 64 else 1                 # super-amostragem só em imagens pequenas
    spec, render = fourier_image(n, IMG_SIZE * up, decay, seed)
    opt = torch.optim.Adam([spec], lr=lr)
    pad = max(1, IMG_SIZE // 8)
    for _ in tqdm(range(steps), leave=False):
        x = F.pad(render(), (pad * up,) * 4, mode="reflect")
        sc = float(rng.choice([0.9, 0.95, 1.0, 1.05, 1.1])) / up
        x = F.interpolate(x, scale_factor=sc, mode="bilinear", align_corners=False, antialias=True)
        dx, dy = rng.integers(-pad // 2, pad // 2 + 1, 2).tolist()
        x = (torch.roll(x, (dx, dy), (2, 3)) - MEAN) / STD
        with Capture(model, [CAMADA], kind="bn", detach=False) as cap:
            logits = model(x)
        loss = -objective(logits, cap).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        img = F.interpolate(render(), size=IMG_SIZE, mode="bilinear", align_corners=False, antialias=True)
    return img.permute(0, 2, 3, 1).cpu().numpy()


def show_grid(imgs, titles, suptitle, name, cols=8):
    rows = int(np.ceil(len(imgs) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.3, rows * 1.45), squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < len(imgs):
            ax.imshow(imgs[i])
            ax.set_title(titles[i], fontsize=7)
    fig.suptitle(suptitle, y=1.02)
    savefig(fig, name)
    plt.show()

In [ ]:
ch_t = torch.tensor(canais, device=device)
feat_imgs = optimize_images(lambda logits, cap: cap[CAMADA][torch.arange(len(canais)), ch_t].mean((1, 2)),
                            n=len(canais))
show_grid(feat_imgs, [f"#{c}" for c in canais], f"{CAMADA}: imagens que maximizam cada canal",
          f"featvis_{CAMADA}")

In [ ]:
cls_t = torch.arange(NUM_CLASSES, device=device)
class_imgs = optimize_images(lambda logits, cap: logits[cls_t, cls_t], n=NUM_CLASSES)
show_grid(class_imgs, [n[:12] for n in class_names], "imagens que maximizam o logit de cada classe", "featvis_classes")

## 6. Grad-CAM

Mapa de relevância para a classe **predita**, calculado em `CAMADA_GRADCAM`:
pesos = média espacial do gradiente do logit em relação às ativações; CAM = ReLU(Σ peso·ativação).
As imagens são escolhidas para cobrir as classes (uma por classe, em ciclo).

In [ ]:
def grad_cam(x, layer, target=None):
    """x: (N,3,H,W) normalizado. Retorna (cams (N,H,W) em [0,1], preds, probs)."""
    x = x.to(device).clone().requires_grad_(True)   # parâmetros estão congelados; o grafo precisa de uma folha com grad
    with Capture(model, [layer], kind="relu", detach=False) as cap:
        logits = model(x)
    preds = logits.argmax(1) if target is None else torch.as_tensor(target, device=device)
    score = logits.gather(1, preds[:, None]).sum()
    act = cap[layer]
    grads = torch.autograd.grad(score, act)[0]
    weights = grads.mean((2, 3), keepdim=True)
    cam = F.relu((weights * act).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-1], mode="bilinear", align_corners=False).squeeze(1)
    cam = cam / (cam.amax((1, 2), keepdim=True) + 1e-8)
    probs = logits.softmax(1).gather(1, preds[:, None]).squeeze(1)
    return cam.detach().cpu().numpy(), preds.cpu().numpy(), probs.detach().cpu().numpy()


# uma imagem por classe, em ciclo
per_class = [np.flatnonzero(labels.numpy() == k) for k in range(NUM_CLASSES)]
pick = [per_class[i % NUM_CLASSES][i // NUM_CLASSES] for i in range(N_IMAGENS_GRADCAM)]
xb = torch.stack([test_ds[i][0] for i in pick])
cams, preds, probs = grad_cam(xb, CAMADA_GRADCAM)
imgs = denorm(xb)

fig, axes = plt.subplots(3, len(pick), figsize=(len(pick) * 1.5, 5.2), squeeze=False)
for c, i in enumerate(pick):
    true = labels[i].item()
    mark = "" if preds[c] == true else " ✗"
    axes[0, c].imshow(imgs[c])
    axes[0, c].set_title(f"real: {class_names[true][:10]}\npred: {class_names[preds[c]][:10]} ({probs[c]:.2f}){mark}",
                         fontsize=7)
    axes[1, c].imshow(cams[c], cmap="gray", vmin=0, vmax=1)
    axes[2, c].imshow(spotlight(imgs[c], cams[c]))
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
axes[1, 0].set_ylabel("Grad-CAM", fontsize=8)
axes[2, 0].set_ylabel("sobreposição", fontsize=8)
fig.suptitle(f"Grad-CAM em {CAMADA_GRADCAM} (classe predita)", y=1.02)
plt.tight_layout()
savefig(fig, f"gradcam_{CAMADA_GRADCAM}")
plt.show()

## 7. Embeddings da penúltima camada

Vetor que entra no último linear (512-d no modelo `"cifar"`, 4096-d no `"imagenet"`), projetado em 2D.
Classes bem separadas aqui indicam que o classificador final tem uma tarefa fácil.

In [ ]:
feats = []
h = model.classifier[-1].register_forward_pre_hook(lambda _m, inp: feats.append(inp[0].detach().cpu()))
with torch.no_grad():
    for xb, _ in tqdm(test_loader, desc="embeddings"):
        model(xb.to(device))
h.remove()
feats = torch.cat(feats).numpy()
lbl = labels.numpy()

emb_pca = PCA(2, random_state=SEED).fit_transform(feats)
emb_tsne = TSNE(2, init="pca", perplexity=min(30, len(feats) // 4), random_state=SEED).fit_transform(feats)
print("features:", feats.shape)

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, emb, title in ((axes[0], emb_pca, "PCA"), (axes[1], emb_tsne, "t-SNE")):
    if NUM_CLASSES <= len(PALETTE):
        for k in range(NUM_CLASSES):
            m = lbl == k
            ax.scatter(emb[m, 0], emb[m, 1], s=10, color=PALETTE[k], alpha=0.75, linewidths=0, label=class_names[k])
    else:  # muitas classes: cor única + rótulo no centroide
        ax.scatter(emb[:, 0], emb[:, 1], s=6, color="#8a8984", alpha=0.5, linewidths=0)
    for k in range(NUM_CLASSES):
        cx, cy = np.median(emb[lbl == k], axis=0)
        ax.annotate(class_names[k], (cx, cy), fontsize=8, ha="center", va="center", color="#0b0b0b",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
if NUM_CLASSES <= len(PALETTE):
    axes[1].legend(fontsize=8, markerscale=2, frameon=False, loc="center left", bbox_to_anchor=(1, 0.5))
fig.suptitle("embeddings da penúltima camada (teste)")
plt.tight_layout()
savefig(fig, "embeddings")
plt.show()